# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [23]:
%pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr arxiv ddgs

Note: you may need to restart the kernel to use updated packages.


In [24]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


### Tools

In [25]:
import importlib, pkgutil

# langchain_community.tools 패키지 로드
package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위모듈들을 하나씩 순회 
for module in pkgutil.iter_modules(package.__path__):
    print(module.name)  # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### Wikipedia Tool

In [26]:
from langchain_community.tools import WikipediaQueryRun     # 위키피디아 질문 실행 tool
from langchain_community.utilities import WikipediaAPIWrapper   # 위키피디아 검색/요약 API 요청 래퍼 클래스

# 위키피디아 API 래퍼를 Tool에 연걸
wiki_tool = WikipediaQueryRun(api_wrapper= WikipediaAPIWrapper())
print(wiki_tool.run('Physical AI')) # 위키피디아 검색/요약 결과 출력

Page: Physical artificial intelligence
Summary: Physical artificial intelligence or physical AI refers to artificial intelligence (AI) systems that perceive, reason about and act within the physical world. These systems generally combine AI models with sensors, control systems, actuators and physical machines such as robots or autonomous vehicles. Physical AI overlaps with embodied artificial intelligence, robotics and autonomous systems, but it emphasizes the complete process of perceiving an environment, motion planning an action and physically executing the task to perform work. This differs from digital AI or generative AI (GenAI), which primarily stays in the information or digital realm.
The term became increasingly prominent during the AI boom in the 2020s as AI development expanded from primarily digital applications toward humanoid robots, self-driving vehicles, smart factories and other autonomous machines. Its boundaries are not standardized, and it is often treated as a con

In [27]:
from langchain.chat_models import init_chat_model   
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '한국그룹 롱샷의 멤버 알려줘')]
llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드 멤버 알려줘'))  # 최신 정보 알지 못함

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages' : messages})

pprint(response)

{'messages': [HumanMessage(content='한국그룹 롱샷의 멤버 알려줘', additional_kwargs={}, response_metadata={}, id='78ef5990-7a53-455e-9593-ba82ab2bc2f9'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 175, 'total_tokens': 196, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPQa2ygbBpZJQemHaueGJC1sdmhX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-2c07-7e01-8719-3cb48ea1c858-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Longshot South Korean group members'}, 'id': 'call_UmTQm

### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [28]:
import requests                         # HTTP 요청 보내는 라이브러리
import xml.etree.ElementTree as ET      # XML 응답 파싱
from langchain_core.tools import tool   # Langchain Tool 생성 데코레이터 

@tool

def search_arxiv(arxiv_id : str) -> str:
    """ arXiv 논문 ID로 제목, 저자, 초록을 조회합니다 """

    url = 'https://export.arxiv.org/api/query'
    response = requests.get(
        url,
        params = {
            'id_list':arxiv_id,     # 논문 ID 
            'max_results':1         # 결과 1개
        },  
        timeout = 10        # 응답 대기시간
    )

    response.raise_for_status() # 요청 실패시 예외 발생

    root = ET.fromstring(response.text) # XML 문자열을 받아 Element 객체로 변환

    ns = {'atom':'http://www.w3.org/2005/Atom'} # arXiv 응답의 XML 네임스페이스
    entry = root.find('atom:entry', ns)     # 논문 정보가 담긴 entry 태그

    if entry is None:
        return '논문 정보를 찾을 수 없습니다.'

    title = entry.findtext('atom:title',namespaces=ns).strip()  # 논문 제목 추출
    summary = entry.findtext('atom:summary', namespaces=ns).strip() # 요약 정보 추출
    authors = [     # 저자 추출
        author.findtext('atom:name',namespaces=ns)
        for author in entry.findall('atom:author',ns)
    ]
    return f""" 
제목 : {title}
저자 : {', '.join(authors)}
초록 : {summary}
"""

In [29]:
tools = [search_arxiv,wiki_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt= '당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.'
)

messages = [('human', '1706.03762 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages':messages})
print(response['messages'][-1].content)

네. arXiv **1706.03762**는 유명한 논문 **“Attention Is All You Need”**로, **Transformer**라는 새로운 신경망 구조를 소개합니다.

### 핵심 내용
- 기존의 번역 모델들은 주로 **RNN**이나 **CNN**을 사용했는데, 이 논문은 그런 구조 없이 **Attention 메커니즘만으로** 모델을 만들 수 있다고 보여줍니다.
- 이 구조를 **Transformer**라고 부릅니다.

### 왜 중요한가?
- **병렬 처리**가 훨씬 잘 돼서 학습이 빠릅니다.
- 기존 모델보다 **성능도 더 좋거나 비슷**한 수준을 냅니다.
- 특히 **기계번역**에서 좋은 결과를 보였습니다.

### 논문의 실험 결과
- 영어→독일어 번역과 영어→프랑스어 번역에서 당시 최고 수준의 성능을 달성했습니다.
- 학습 시간도 기존보다 훨씬 적게 들었습니다.

### 한 줄 요약
**“문장을 순서대로 처리하는 RNN 대신, attention만으로도 번역을 잘할 수 있고 더 빠르게 학습되는 Transformer를 제안한 논문”**입니다.

원하시면 제가 이 논문을 **그림 없이도 쉽게 이해되도록**, 또는 **Self-Attention / Multi-Head Attention 중심으로 더 자세히** 설명해드릴게요.


### LLM-math

In [30]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')

# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm 필요)
tools = load_tools(['wikipedia','llm-math'],llm=llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt= '''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요.
단, 숫자계산은 반드시 llm-math 도구를 사용해서 답변에 활용해야 합니다.
    '''
)

response = agent.invoke({'messages':'3.5의3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘'})
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='3.5의3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘', additional_kwargs={}, response_metadata={}, id='f696e529-0d10-4426-9c5e-51b836978da4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 188, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8d6b6bc837', 'id': 'chatcmpl-EHPQm6lsUnP90ykbgXKVPBf37ipIk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-5d61-7cc3-985f-914d1d4a61c6-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '3.5^3'}, 'id': 'call_yYbZIKU

### duckduckgo

https://reference.langchain.com/python/langchain-community/tools/ddg_search/tool/DuckDuckGoSearchRun  

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [31]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun()    # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke('trumps first name?'))    # 문자열 출력

ddgs2 = DuckDuckGoSearchResults()   # 검색 결과를 제목/링크/스니펫 형태로 반환
print(ddgs2.invoke('trumps first name?'))   # 결과 출력

2.2Licensing the Trump name. 2.3Side ventures. 2.3.1Trump University. Donald Trump 's first tenure as the president of the United States began on January 20, 2017, when he was inaugurated as the 45th president, and ended on January 20, 2021. Trump, a Republican, took office after defeating Democratic nominee Hillary Clinton in 2016. Donald Trump (born June 14, 1946, New York, New York, U.S.) 45th president of the United States (2017–21). Trump was a real estate developer and businessman who owned, managed, or licensed his name to hotels, casinos, golf courses, resorts, and residential properties in the New York City area and around the world. From the 1980s Trump also lent his name to scores of retail ventures—including branded lines of clothing, cologne, food, and furniture—and to Trump University, which offered seminars in real estate education from 2005 to 2010. In the early 21st century his private conglomerate, the Trump Organization, comprised some 500 companies involved in a wid

In [32]:
llm = init_chat_model('gpt-4.1-mini')

# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm 필요)
tools = [ddgs2]

agent = create_agent(llm,tools)

response = agent.invoke({'messages':'gs25 민음사 빵'})
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='gs25 민음사 빵', additional_kwargs={}, response_metadata={}, id='f23af825-42f6-4a49-a276-e428dfe7328a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 81, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_ddbb4fb8e1', 'id': 'chatcmpl-EHPQxEofbU6bDQvNQQ10ut7oR3WhX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-8835-7173-b3cf-7a791ea73c4e-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'gs25 민음사 빵'}, 'id': 'call_GpHSkjFVI1J

### tavily-search

https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [33]:
from langchain_tavily import TavilySearch   # Tavily 검색 tool

tavily_tool = TavilySearch(
    max_results= 3,
    topic = 'general',  # general/news/finance 등 선택
    include_images= True,
    search_depth= 'advanced'
)
tavily_tool.invoke('2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?')

{'query': '2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426f67d1b5fa839e454dfe_79_thumbnail2.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426e7fc594ca778cc36ab8_79_2-1.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426ed2d1b5fa839e44e068_79_2-2.png',
  'https://cdn.wakeupnews.co.kr/news/photo/202601/973_1856_5154.png',
  'https://i.ytimg.com/vi/spk_eNxCH_k/maxresdefault.jpg'],
 'results': [{'url': 'https://www.newsshin.co.kr/news/articleView.html?idxno=325554',
   'title': "【뉴스신ㅣ2026년 8월 22일(토) ㅣ대한민국 '핫' 이슈】 < 기자수첩 < 뉴스신 광장 < 기사본문 - 뉴스신(NEWSSHIN)",
   'content': '☞ 뉴스신 기자의 시선  \n 국민은 경기 결과보다 그 안에서 희망을 찾는다.\n\n▣ 금융·가계  \n "빚의 시대, 관리 능력이 생존력"  \n ▶ 브리핑  \n 가계부채와 금리 부담은 여전히 경제의 핵심 변수다.  \n → 구조 분석  \n 금융 안정은 국가 경제 안정과 직결된다.\n\n☞ 뉴스신 기자의 시선  \n 돈을 버는 능력만큼 중요한 시대가 왔다.  \n 돈을 지키는 능력이다.\n\n▣ 미래 대한민국  \n "AI 시대 국가 경쟁력은 사람이다"  \n ▶ 브리핑

In [34]:
tools = [tavily_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt= '''
당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)
'''
)
response = agent.invoke({'messages': '2026년 메타 주식 분석해줘'})
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='2026년 메타 주식 분석해줘', additional_kwargs={}, response_metadata={}, id='3de18768-b615-4f98-9e39-46dba8dab342'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 1295, 'total_tokens': 1325, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a54630b043', 'id': 'chatcmpl-EHPR79ga4pQVQHcu6EIhz86dGotYU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-ae78-7be2-800d-8d84c66cfb12-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Meta stock analysis 2026', 'time_range

In [35]:
from IPython.display import display, Markdown

# 마지막 메시지 내용을 Markdown 변환 후 보기 좋게 출력
display(Markdown(response['messages'][-1].content))

2026년 메타(META) 주식에 대한 주요 분석기관들의 전망을 정리하면 다음과 같습니다.

| 분석기관명               | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드                                      | 신뢰도 지수 (1~10) |
|-----------------------|-----------------------|-------------------------------------------------|-----------------|
| Vantage Markets       | $580 ~ $1,000          | 광고수익 증가, 일일 활성 사용자 증가, AI 투자 확대, 비용 증가와 수익성 압박       | 8               |
| TradingKey            | $540 ~ $700            | 광고 매출 성장, AI 도구로 인한 광고 전환율 상승, AI 인프라 투자 부담, 단기 하락 압력 | 7               |
| 24/7 Wall St.         | $700 ~ $935            | AI 투자에 따른 효율성 개선, 강한 광고 매출, 수익성 개선 기대                     | 8               |

요약:
- 메타 광고 매출이 크게 증가하며 일일 활성 사용자도 성장 중입니다.
- 인공지능(AI) 기술 도입으로 광고 효율성이 상승해 수익 개선 기대가 있지만,
- AI 인프라 투자로 비용이 크게 늘어나면서 단기적으로 수익성 압박과 주가 변동성이 존재합니다.
- 다양한 목표주가가 제시되는데, 이는 AI 투자에 따른 불확실성과 성장 잠재력을 반영합니다.

투자 결정 시 AI 투자 집행 상황, 광고 시장 동향 및 비용 구조 변화를 주시하는 것이 중요합니다.

# @tool

In [36]:
# eval / exec로 문자열 코드 실행
a = 10
print(eval('5+3+a'))    # 문자열을 평가해서 결과를 반환
exec('b=10')
print(b)

18
10


In [37]:
from langchain_core.tools import tool

@tool

def simple_calculator(query:str) -> str:
    """산술연산을 위한 간단한 계산기 Tool"""  # 함수 설명(1줄)
    """
    산술연산을 위한 간단한 계산기
    Args:
        query: 계산식
    Return:
        계산식 결과값

    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"
    """
    try:
        result = eval(query)            # 문자열을 eval로 평가(결과 반환)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류 : {str(e)}"

simple_calculator

StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x00000113D34DC9A0>)

In [38]:
llm = init_chat_model('gpt-5.4-mini')

tools = [simple_calculator]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt= '''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.
'''
)
response = agent.invoke({'messages': '7+3*8 이거를 계산해 줘.'})
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='7+3*8 이거를 계산해 줘.', additional_kwargs={}, response_metadata={}, id='983e3ac7-0c82-4a6c-bab9-cb161b9ca4dd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 185, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPRF1eI1ExyM57vDiElVxQR9PVHU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-ce55-7a83-99ec-3d4220b7e67d-0', tool_calls=[{'name': 'simple_calculator', 'args': {'query': '7+3*8'}, 'id': 'call_HGiTbt3LxHJRtlAeN19XrwI4',

In [39]:
response = agent.invoke({'messages': '김치볶음밥 레시피?'})
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='김치볶음밥 레시피?', additional_kwargs={}, response_metadata={}, id='c88cd0ed-86b1-4410-ba0f-77e883fa370f'),
              AIMessage(content='물론이죠. 간단하고 맛있는 **김치볶음밥 레시피** 알려드릴게요.\n\n## 재료 1~2인분\n- 밥 2공기\n- 김치 1컵 정도\n- 김치국물 2~3큰술\n- 양파 1/4개\n- 대파 조금\n- 식용유 1큰술\n- 참기름 1큰술\n- 설탕 1작은술\n- 간장 1작은술\n- 고춧가루 1작은술(선택)\n- 달걀 1~2개\n- 김가루, 깨 약간\n\n## 만드는 법\n1. **재료 손질**\n   - 김치는 잘게 썰고, 양파와 대파도 다져 주세요.\n\n2. **볶기**\n   - 팬에 식용유를 두르고 대파를 먼저 볶아 향을 냅니다.\n   - 양파를 넣고 살짝 볶은 뒤 김치를 넣어 같이 볶아 주세요.\n\n3. **양념 넣기**\n   - 설탕, 간장, 고춧가루를 넣고 볶습니다.\n   - 김치국물도 넣으면 더 감칠맛이 납니다.\n\n4. **밥 넣기**\n   - 밥을 넣고 잘 풀어가며 볶습니다.\n   - 밥이 고르게 섞이면 불을 약하게 줄여 1~2분 더 볶아 주세요.\n\n5. **마무리**\n   - 참기름을 넣고 한 번 더 섞은 뒤 불을 끕니다.\n\n6. **토핑**\n   - 계란프라이를 올리고 김가루, 깨를 뿌리면 완성입니다.\n\n## 팁\n- **신 김치**를 쓰면 더 맛있어요.\n- 밥은 **찬밥**이 볶기 좋아요.\n- 더 고소하게 먹고 싶으면 **버터 조금** 넣어도 맛있습니다.\n\n원하시면 제가 **초간단 5분 버전**이나 **스팸 넣는 버전**도 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 477,

In [40]:
import json

OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')


In [41]:

@tool
def get_current_weather(city = 'Seoul', units='metric'):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
            - 변환예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도단위를 설정하는 문자열
          - metric(기본값: 섭씨, 미터)
          - imperial(화씨, 야드)
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """

    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json() # json -> dict

    weather_info = {}

    if response.status_code == 200: # 정상 응답 받은 경우
        weather_description = data['weather'][0]['description'] # 날씨 설명
        temp = data['main']['temp'] # 현재 기온
        temp_fells_like = data['main']['feels_like']    # 체감 온도
        humidity = data['main']['humidity']     # 습도

        weather_info = {
            'city' : city,
            'description' : weather_description,
            'temperature' : temp,
            'temperature_feels_like' : temp_fells_like,
            'humidity' : humidity
        }
    else:   # 응답 불량
        weather_info = {
            'city' : city,
            'description' : "Not Found",
            'temperature' : "Not Found",
            'temperature_feels_like' : "Not Found",
            'humidity' : "Not Found"
        }
    return json.dumps(weather_info)
get_current_weather

StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**\n        - 변환예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도단위를 설정하는 문자열\n      - metric(기본값: 섭씨, 미터)\n      - imperial(화씨, 야드)\nReturn:\n    - str: json 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x00000113D3597F60>)

In [42]:
llm = init_chat_model('gpt-5.4-mini')

tools = [simple_calculator,get_current_weather]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt= '''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.
'''
)
response = agent.invoke({'messages': '오늘 뭐 입어야 돼? 나 서울 살아.'})
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='오늘 뭐 입어야 돼? 나 서울 살아.', additional_kwargs={}, response_metadata={}, id='d54d01d7-310f-4a5a-91dc-f912e3b748be'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 352, 'total_tokens': 375, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPRJv4HGIZGAXUO6EYUIIFhYMINZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-de66-7253-9725-21a458a342e6-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Seoul', 'units': 'metric'}, 'id': 'call_zJ

In [43]:
# 한국 기준 현재 날짜/시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone

@tool

def get_current_datetime(format:str='%Y-%,-%d %H:%M:%S')->str:
    """
    한국기준 현재시각정보를 반환하는 함수
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """

    kst = timezone('Asia/Seoul')    # 한국 시간대(KST) 설정
    return datetime.now(kst).strftime(format)   # 현재 서울 시간을 받아, format 형식의 문자열로 반환
get_current_datetime

StructuredTool(name='get_current_datetime', description='한국기준 현재시각정보를 반환하는 함수\nArgs:\n    format: 날짜/시각 형식 지정\nReturn:\n    현재시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x00000113D3594C20>)

In [44]:
@tool
def calculate_age(today_date : str, birth_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """
    try:
        today = datetime.strptime(today_date, '%Y-%m-%d')
        birthday = datetime.strptime(birth_date, '%Y-%m-%d')

        age = today.year - birthday.year
        if (today.month, today.day) < (birthday.month,birthday.day):
            age-=1
        return age
    except ValueError:
        return '날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해 주세요.'

calculate_age.invoke({'today_date': '2026-08-27', 'birth_date':'2020-02-16'})

6

In [56]:
llm = init_chat_model('gpt-5.4-mini')

tools = load_tools(['wikipedia']) + [get_current_datetime,calculate_age]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt= '''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.
'''
)
response = agent.invoke(
    {'messages': ['human','트럼프 대통령의 현재 나이는?']},
    config = {'recursion_limit': 10}
)
pprint(response)
print('='*50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='human', additional_kwargs={}, response_metadata={}, id='88596565-c23c-4148-b071-20520ed6449e'),
              HumanMessage(content='트럼프 대통령의 현재 나이는?', additional_kwargs={}, response_metadata={}, id='79f42551-920f-4f95-91b7-5cf35eef9ebe'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 382, 'total_tokens': 405, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPUKAGnyKgYOSIMMcFrU80F5uHdt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04245-b94e-741

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

llm = init_chat_model('gpt-5.4-mini')

tools = [TavilySearch()]

# 체크포인터를 전달하여 대화 상태 저장 가능하도록 에이전트 생성
agent = create_agent(llm, tools, checkpointer=InMemorySaver())

response = agent.invoke(
    input = {'messages':[('human', '안녕! 만나서 반갑다! 나는 cap이라고 해. 넌 누구니?')]},
    config={'configurable':{'thread_id':'100'}}     # thread_id로 대화 식별
)

print(response['messages'][-1].content)

안녕 cap! 만나서 반가워 🙂  
나는 OpenAI가 만든 AI 어시스턴트야. 질문 답변, 글쓰기, 번역, 아이디어 정리 같은 걸 도와줄 수 있어.

편하게 불러줘!


In [ ]:
response = agent.invoke(
    input = {'messages':[('human', '어 그래 너 GPT구나~ 내이름이 뭐였지? 나 기억상실증이야!')]},
    config={'configurable':{'thread_id':'100'}} # thread_id 100번으로 대화 유지 
)

print(response['messages'][-1].content)

네 이름은 **cap**이야 🙂


In [ ]:
response = agent.invoke(
    input = {'messages':[('human', '어 그래 너 GPT구나~ 내이름이 뭐였지? 나 기억상실증이야!')]},
    config={'configurable':{'thread_id':'200'}} # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

지금 대화만으로는 네 이름을 알 수 없어.  
내가 이전 대화 내용을 기억하는 기능은 없어서, 네가 직접 알려줘야 해!

원하면 내가 이렇게 도와줄게:
- 네 이름을 기억하기 쉽게 정리해주기
- 별명으로 불러주기
- “내 이름은 ○○야”라고 말하면 그걸 기준으로 대화하기

이름 한번만 알려줘 🙂


In [60]:
response = agent.invoke(
    input = {'messages':[('human', '내이름은 아무것도 아니다!')]},
    config={'configurable':{'thread_id':'200'}} # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

알겠어, **아무것도 아니다**라고 부를게 😄  
그럼 나도 편하게 이야기할게. 무엇을 도와줄까?


In [61]:
response = agent.invoke(
    input = {'messages':[('human', '내이름이 뭐라고?')]},
    config={'configurable':{'thread_id':'200'}} # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

네 이름은 **아무것도 아니다**야.


# sqliteSaver

In [ ]:
# langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')

tools = [TavilySearch()]

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer= checkpointer)

    response = agent.invoke(
        input = {'messages':[('human', 'Langchain에 대해 설명해줘')]},
        config={'configurable':{'thread_id':'100'}}     # thread_id로 대화 식별
    )

    pprint(response)
    print('='*50)
    print(response['messages'][-1].content)

    print('='*100)
    response = agent.invoke(
        input = {'messages':[('human', 'Langgraph에 대해 설명해줘')]},
        config={'configurable':{'thread_id':'100'}}     # thread_id로 대화 식별
    )    

    pprint(response)
    print('='*50)
    print(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘', additional_kwargs={}, response_metadata={}, id='76e0fdb7-f4ac-4cf7-8507-e2c2b50d29a2'),
              AIMessage(content='LangChain은 **LLM(대규모 언어 모델)**을 활용한 애플리케이션을 쉽게 만들 수 있게 도와주는 **오픈소스 프레임워크**입니다.  \n한마디로 말하면, **챗봇, 검색형 QA, 에이전트, 문서 요약, RAG 시스템** 같은 걸 빠르게 구축하도록 돕는 도구 모음이에요.\n\n## LangChain이 왜 필요한가?\nLLM은 문장을 잘 생성하지만, 실제 서비스에 쓰려면 보통 이런 기능이 필요합니다.\n\n- 외부 데이터와 연결\n- 여러 단계의 작업 처리\n- 문서 검색\n- API 호출\n- 메모리 유지\n- 도구 사용\n- 여러 모델/벡터DB/데이터소스 연동\n\nLangChain은 이런 작업을 **체인(chain)**처럼 엮어서 처리할 수 있게 해줍니다.\n\n---\n\n## 주요 개념\n\n### 1. Chains\n여러 작업을 순서대로 연결한 것  \n예:\n- 사용자 질문 입력\n- 관련 문서 검색\n- 검색 결과를 바탕으로 답변 생성\n\n### 2. Models\nOpenAI, Anthropic, Hugging Face 등 다양한 LLM을 연결할 수 있습니다.\n\n### 3. Prompts\n모델에게 어떤 형식으로 답하게 할지 템플릿을 관리합니다.\n\n### 4. Retrievers\n문서나 데이터베이스에서 관련 정보를 찾아오는 역할입니다.  \nRAG에서 핵심입니다.\n\n### 5. Vector Stores\n문서를 임베딩해서 저장하고, 유사도 검색을 하게 해줍니다.  \n예: FAISS, Chroma, Pinecone, Weaviate\n\n### 6. Agents\n모델이 상황에 따라 어떤 도구를 쓸지 스스로 결정

In [66]:
from langgraph.checkpoint.sqlite import SqliteSaver
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')

tools = [TavilySearch()]

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer= checkpointer)

    response = agent.invoke(
        input = {'messages':[('human', '오케이 완전 이해했어 그럼 니가 말해준 설명을 세줄로 요약해줘')]},
        config={'configurable':{'thread_id':'100'}}     # thread_id로 대화 식별
    )

    pprint(response)
    print('='*50)
    print(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘', additional_kwargs={}, response_metadata={}, id='76e0fdb7-f4ac-4cf7-8507-e2c2b50d29a2'),
              AIMessage(content='LangChain은 **LLM(대규모 언어 모델)**을 활용한 애플리케이션을 쉽게 만들 수 있게 도와주는 **오픈소스 프레임워크**입니다.  \n한마디로 말하면, **챗봇, 검색형 QA, 에이전트, 문서 요약, RAG 시스템** 같은 걸 빠르게 구축하도록 돕는 도구 모음이에요.\n\n## LangChain이 왜 필요한가?\nLLM은 문장을 잘 생성하지만, 실제 서비스에 쓰려면 보통 이런 기능이 필요합니다.\n\n- 외부 데이터와 연결\n- 여러 단계의 작업 처리\n- 문서 검색\n- API 호출\n- 메모리 유지\n- 도구 사용\n- 여러 모델/벡터DB/데이터소스 연동\n\nLangChain은 이런 작업을 **체인(chain)**처럼 엮어서 처리할 수 있게 해줍니다.\n\n---\n\n## 주요 개념\n\n### 1. Chains\n여러 작업을 순서대로 연결한 것  \n예:\n- 사용자 질문 입력\n- 관련 문서 검색\n- 검색 결과를 바탕으로 답변 생성\n\n### 2. Models\nOpenAI, Anthropic, Hugging Face 등 다양한 LLM을 연결할 수 있습니다.\n\n### 3. Prompts\n모델에게 어떤 형식으로 답하게 할지 템플릿을 관리합니다.\n\n### 4. Retrievers\n문서나 데이터베이스에서 관련 정보를 찾아오는 역할입니다.  \nRAG에서 핵심입니다.\n\n### 5. Vector Stores\n문서를 임베딩해서 저장하고, 유사도 검색을 하게 해줍니다.  \n예: FAISS, Chroma, Pinecone, Weaviate\n\n### 6. Agents\n모델이 상황에 따라 어떤 도구를 쓸지 스스로 결정

In [67]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({'configurable':{'thread_id':'100'}})

    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']

    for i, message in enumerate(messages,1):
        msg_type = getattr(message,'type',message.__class__.__name__)
        print(f"{i}:[{msg_type}]{message.content}")
        print()

1:[human]Langchain에 대해 설명해줘

2:[ai]LangChain은 **LLM(대규모 언어 모델)**을 활용한 애플리케이션을 쉽게 만들 수 있게 도와주는 **오픈소스 프레임워크**입니다.  
한마디로 말하면, **챗봇, 검색형 QA, 에이전트, 문서 요약, RAG 시스템** 같은 걸 빠르게 구축하도록 돕는 도구 모음이에요.

## LangChain이 왜 필요한가?
LLM은 문장을 잘 생성하지만, 실제 서비스에 쓰려면 보통 이런 기능이 필요합니다.

- 외부 데이터와 연결
- 여러 단계의 작업 처리
- 문서 검색
- API 호출
- 메모리 유지
- 도구 사용
- 여러 모델/벡터DB/데이터소스 연동

LangChain은 이런 작업을 **체인(chain)**처럼 엮어서 처리할 수 있게 해줍니다.

---

## 주요 개념

### 1. Chains
여러 작업을 순서대로 연결한 것  
예:
- 사용자 질문 입력
- 관련 문서 검색
- 검색 결과를 바탕으로 답변 생성

### 2. Models
OpenAI, Anthropic, Hugging Face 등 다양한 LLM을 연결할 수 있습니다.

### 3. Prompts
모델에게 어떤 형식으로 답하게 할지 템플릿을 관리합니다.

### 4. Retrievers
문서나 데이터베이스에서 관련 정보를 찾아오는 역할입니다.  
RAG에서 핵심입니다.

### 5. Vector Stores
문서를 임베딩해서 저장하고, 유사도 검색을 하게 해줍니다.  
예: FAISS, Chroma, Pinecone, Weaviate

### 6. Agents
모델이 상황에 따라 어떤 도구를 쓸지 스스로 결정하게 하는 방식입니다.  
예:
- 계산이 필요하면 계산기 사용
- 최신 정보가 필요하면 검색 API 사용

### 7. Memory
대화 내용을 기억하게 해주는 기능입니다.  
챗봇에서 이전 대화를 반영할 때 사용합니다.

---

## 대표적인 활용 사례

- **문서 기반 QA**
- **사내 지식 검색 챗봇**
- **RA